In [1]:
# Needed imports
#SOM example code by Greta Easthom, mostly modeled off of Dr. Maria Molina's SOM codes
from minisom import MiniSom, asymptotic_decay
import xarray as xr
import cftime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
import pickle

In [2]:
#moisture=xr.open_dataset('wwrfss_matched_final.nc')

In [3]:
#moisture

In [4]:
!wget https://github.com/geastho/AtmosphericRiver_predictive_skill_2025/raw/main/regridwrfss_matched_0605.nc
#Greta Easthom's github

moisture = xr.open_dataset('regridwrfss_matched_0605.nc',engine='netcdf4')

# # Step 3: Explore the dataset
# print(ds)
#Greta Easthom's github


--2025-04-18 12:26:54--  https://github.com/geastho/AtmosphericRiver_predictive_skill_2025/raw/main/regridwrfss_matched_0605.nc
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/geastho/AtmosphericRiver_predictive_skill_2025/main/regridwrfss_matched_0605.nc [following]
--2025-04-18 12:26:55--  https://raw.githubusercontent.com/geastho/AtmosphericRiver_predictive_skill_2025/main/regridwrfss_matched_0605.nc
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22127384 (21M) [application/octet-stream]
Saving to: ‘regridwrfss_matched_0605.nc.3’

regridwrfss_matched 100%[===================>]  21.10M   109M

In [5]:
# Step 1: Download the file
!wget https://github.com/geastho/AtmosphericRiver_predictive_skill_2025/raw/main/ht_matched_final.nc
#Greta Easthom's github

height = xr.open_dataset('ht_matched_final.nc',engine='netcdf4')

# # Step 3: Explore the dataset
# print(ds)

--2025-04-18 12:26:55--  https://github.com/geastho/AtmosphericRiver_predictive_skill_2025/raw/main/ht_matched_final.nc
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/geastho/AtmosphericRiver_predictive_skill_2025/main/ht_matched_final.nc [following]
--2025-04-18 12:26:56--  https://raw.githubusercontent.com/geastho/AtmosphericRiver_predictive_skill_2025/main/ht_matched_final.nc
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21315592 (20M) [application/octet-stream]
Saving to: ‘ht_matched_final.nc.3’

ht_matched_final.nc 100%[===================>]  20.33M   109MB/s    in 0.2s    

2025-04-18 1

In [6]:
height

<xarray.Dataset>
Dimensions:   (y: 56, x: 56, time: 849)
Coordinates:
    lon       (y) float64 ...
    lat       (x) float64 ...
  * time      (time) datetime64[ns] 1985-12-08 1985-12-10 ... 2017-03-31
Dimensions without coordinates: y, x
Data variables:
    variable  (time, y, x) float64 ...

In [7]:
mask=moisture['time'].isin(height['time']) #mask the times just to make sure all times line up

In [8]:
moisture['variable'][mask]

<xarray.DataArray 'variable' (time: 849, x: 56, y: 56)>
[2662464 values with dtype=float64]
Coordinates:
    lon      (time, y) float64 ...
    lat      (time, x) float64 ...
  * time     (time) datetime64[ns] 1985-12-08 1985-12-10 ... 2017-03-31
Dimensions without coordinates: x, y

In [9]:
subsetarray = moisture['variable'][mask].stack(new=("x","y")) #condense 3d variable into a 2d variable
print(subsetarray.shape) # check the dims/shape, should be time, lat*lon

(849, 3136)


In [10]:
subsetarray

<xarray.DataArray 'variable' (time: 849, new: 3136)>
array([[192.76066589, 190.63644409, 191.8585968 , ..., 116.17114258,
        120.11898041, 123.25020599],
       [486.93515015, 492.05728149, 430.01855469, ..., 654.37322998,
        660.14343262, 661.40924072],
       [ 96.930336  ,  91.86048126, 104.87213898, ..., 123.07192993,
        120.25973511, 118.9688797 ],
       ...,
       [470.96768188, 438.46939087, 406.70404053, ..., 536.32574463,
        542.81768799, 544.42181396],
       [523.18798828, 501.16268921, 455.99215698, ..., 119.86746216,
        118.44031525, 118.18891907],
       [866.40661621, 837.88568115, 789.20941162, ..., 722.14837646,
        725.99835205, 733.04309082]])
Coordinates:
    lon      (time, new) float64 -159.5 -159.2 -159.0 ... -142.2 -142.0 -141.8
    lat      (time, new) float64 30.25 30.25 30.25 30.25 ... 48.75 48.75 48.75
  * time     (time) datetime64[ns] 1985-12-08 1985-12-10 ... 2017-03-31
  * new      (new) object MultiIndex
  * x        (new) int64 0 0 0 0 0 0 0 0 0 0 0 ... 55 55 55 55 55 55 55 55 55 55
  * y        (new) int64 0 1 2 3 4 5 6 7 8 9 ... 46 47 48 49 50 51 52 53 54 55

In [11]:
# SOM hyperparameters

som_grid_rows = 4           # (y-axis)
som_grid_columns = 4        # (x-axis)
input_length = subsetarray.shape[1]    # using preprocessed data array; Number of the elements of the vectors in input.
sigma = 3               # Spread of the neighborhood function, needs to be adequate to the dimensions of the map.
learning_rate = 0.5       # initial learning rate (at the iteration t we have learning_rate(t) = learning_rate / (1 + t/T) where T is #num_iteration/2)
decay_function = asymptotic_decay

"""Function that reduces learning_rate and sigma at each iteration
    the default function is (asymptotic_decay):
                learning_rate / (1+t/(max_iterarations/2))

    A custom decay function will need to to take in input
    three parameters in the following order:

    1. learning rate
    2. current iteration
    3. maximum number of iterations allowed

    Note that if a lambda function is used to define the decay
    MiniSom will not be pickable anymore."""

neighborhood_function = 'gaussian'

"""Function that weights the neighborhood of a position in the map. 
    Possible values: 'gaussian', 'mexican_hat', 'bubble', 'triangle', 
    which takes in sigma."""

topology = 'rectangular'                 # Topology of the map; Possible values: 'rectangular', 'hexagonal'
activation_distance = 'euclidean'        # Distance used to activate the map; Possible values: 'euclidean', 'cosine', 'manhattan', 'chebyshev'
random_seed = 1                          # Random seed to use for reproducibility. Using 1.

In [12]:
# initialization of SOM

som = MiniSom(
            som_grid_rows,
            som_grid_columns,
            input_length,
            sigma,
            learning_rate,
            decay_function,
            neighborhood_function,
            topology,
            activation_distance,
            random_seed) 

In [13]:
def normalize_data(data):
    """
    Function for normalizing data prior to training using z-score
    """
    return (data - np.nanmean(data)) / np.nanstd(data)

In [14]:
# training hyperparameters

data = normalize_data(subsetarray)
num_iteration = 100000
random_order = True
verbose = True
data.shape

(849, 3136)

In [15]:
# before training, initialize the weights, here using random weights to initialize

# som.random_weights_init(data)    # random method

som.pca_weights_init(data.values)   # or use this: Initializes the weights to span the first two principal components

In [ ]:
# train the SOM 

som.train(
        data.values,
        num_iteration,
        random_order,
        verbose)

 [  84394 / 100000 ]  84% - 0:00:02 left 

In [ ]:
# just checking winners of random data points to make sure they vary

print(som.winner(data.values[10]))
print(som.winner(data.values[0]))
print(som.winner(data.values[100]))
print(som.winner(data.values[5]))

In [ ]:
# view the som lattice's distance map of the weights

"""
Each cell is the normalised sum of the distances between
a neuron and its neighbours. Note that this method uses
the euclidean distance.
"""

plt.figure(figsize=(10, 9))
cs = plt.pcolormesh(som.distance_map(), cmap='bone_r')
plt.title("distance map of the weights", fontsize=12)
plt.colorbar(cs)
plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
plt.show()

In [ ]:
# this may take a few minutes if lots of data! currently doing every 20th point so not bad

# grab the x and y coords across the lattice for the winner node for each data point

w_x, w_y = zip(*[som.winner(d) for d in data.values])
w_x = np.array(w_x)
w_y = np.array(w_y)

# visualize where data falls in lattice (nearest winning neuron/node)

plt.figure(figsize=(10, 9))
plt.pcolormesh(som.distance_map(), cmap='bone_r', alpha=.2)
plt.colorbar()

for num, c in enumerate(w_x[::20]):     # every 20th data point
     
    plt.scatter(w_y[num] + np.random.rand(1),   # add a random decimal to prevent overlapping of markers
                w_x[num] + np.random.rand(1),
                s=50, c='k', marker='x')

plt.title("data across SOM lattice", fontsize=12)
plt.margins(x=0,y=0)
plt.grid()
plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
plt.show()

In [ ]:
# i like this plot -- shows frequencies across lattice

plt.figure(figsize=(10, 9))
frequencies = som.activation_response(data.values)
plt.pcolormesh(frequencies, cmap='Blues') 
plt.colorbar()
plt.title("data frequency (2d histogram) across SOM lattice", fontsize=12)
plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
plt.show()

In [ ]:
# grabbing indices from SOM

# create an empty dictionary using the rows and columns of SOM
keys = [i for i in product(range(som_grid_rows),range(som_grid_columns))]
winmap = {key: [] for key in keys}

# grab the indices for the data within the SOM lattice
for i, x in enumerate(data.values):
    winmap[som.winner(x)].append(i)

In [ ]:
print(f"The rows and columns of the SOM lattice to use to grab SOM indexes:\n{[i for i in list(winmap.keys())]}")

In [ ]:
# create list of the dictionary keys

som_keys = list(winmap.keys())
print(f"Number of composite maps: {len(som_keys)}")

In [ ]:
# view one map

moisture['variable'].stack(new=("y","x"))[np.array(winmap[som_keys[0]])].unstack().mean(dim="time",skipna=True).plot.pcolormesh("y","x",cmap="Blues",vmin=200,vmax=850)

In [ ]:
fig, axs = plt.subplots(som_grid_rows, som_grid_columns, figsize=(16,12))
di_dates={}


##plot som##
#fig, geo_axs = plt.subplots(4, 4, figsize=(16,12))




# (1) loop through the SOM neurons, (2) grab data samples using SOM indices, (3) compute the mean for each SOM neuron, plot each 
for map_num in range(len(som_keys)):

#     
    temp_data = subsetarray[np.array(winmap[som_keys[map_num]])].unstack().mean(dim="time",skipna=True)
    # plot
    ht_data=height['variable'][np.array(winmap[som_keys[map_num]])].mean(dim="time",skipna=True).values
    label=axs[som_keys[map_num][0],som_keys[map_num][1]].pcolormesh(temp_data, cmap="viridis",vmin=200,vmax=850)
    htctr=axs[som_keys[map_num][0],som_keys[map_num][1]].contour(ht_data, colors="white",levels=[5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,6040,6100,6160,6220])
    #levels=[5280,5340,5400,5460,5520,558
    axs[som_keys[map_num][0],som_keys[map_num][1]].set_title(f"Sample size: {frequencies.flatten()[map_num]}", fontsize=12)
    axs[som_keys[map_num][0],som_keys[map_num][1]].clabel(htctr,htctr.levels[::2],fmt='%1.0f',colors='white',inline=True,inline_spacing=-3)


      


    n=(frequencies.flatten()[map_num])




plt.tight_layout()
fig = plt.gcf()
# twin_ax = fig.add_axes([-.05, 0.1, 0.02, 0.8])
# twin_ax.set_ylabel('Gridpoints in the north-south direction (1800 km)', fontsize=20, labelpad=10)
#twin_ax.yaxis.set_label_coords(1.1, 0.5)  # Adjust the position of the label
fig.text(-0.03, 0.5, 'Gridpoints in the south-north direction', fontsize=20, ha='center', va='center', rotation=90)


fig.text(.4,-.05,'Gridpoints in the west-east direction', fontsize=20, ha='center', va='center')

fig.suptitle("Current IVT Shaded \n 500-hPa heights contoured".format(sigma=newsom._sigma,lr =newsom._learning_rate), fontsize=20,y=1.05, x=0.45, fontweight='bold')

#Sigma:{sigma:.2f},LR:{lr:.4f}
#ax1=fig.colorbar(label, ax=fig.get_axes())
cbar=fig.colorbar(label, ax=fig.get_axes())

# cbar.ax.set_ylabel('Difference of IVT btw West-WRF - ERA5 (kg/ms)', rotation=270, fontsize=20,labelpad=20
#       )


cbar.ax.set_ylabel('IVT (kg/ms) shaded', rotation=270, fontsize=20,labelpad=23
      )
plt.savefig('som_ivt_hts.png',bbox_inches='tight',pad_inches = 0)
#plt.savefig('currentIVT_march6_2025.png',bbox_inches='tight',pad_inches = 0)
plt.show()


### organized via more curved heights on the left-hand side to more zonal heights on the right-hand side, intensity on the vertical axis
###

In [ ]:

with open('som_ivt_example.p', 'wb') as outfile:
    pickle.dump(som, outfile)

In [ ]:
#How to load in SOM and example SOM code ensuing:
# with open('som_example_class.p', 'wb') as outfile:
#     pickle.dump(newsom, outfile)
with open('som_ivt_example.p', 'rb') as infile:
    newsom = pickle.load(infile)


In [ ]:
#example SOM code after loading SOM:

som_grid_rows=4
som_grid_columns=4
#data2 = normalize_data(ht_anom_stack.values)
data = normalize_data(subsetarray)
# if som==newsom:
# with open('som_current_mpas_dec5.p', 'wb') as outfile:
#     pickle.dump(newsom, outfile)

# som_grid_rows=4
# som_grid_columns=4

plt.figure(figsize=(10, 9))
cs = plt.pcolormesh(newsom.distance_map(), cmap='hot')
plt.title("distance map of the ivt", fontsize=12)
plt.colorbar(cs)
plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
plt.show()


# this may take a few minutes if lots of data! currently doing every 20th point so not bad

# grab the x and y coords across the lattice for the winner node for each data point

w_x, w_y = zip(*[newsom.winner(w) for w in data.values])
print(len(w_x))
w_x = np.array(w_x)
w_y = np.array(w_y)
print(w_x)
print(w_y)

            # visualize where data falls in lattice (nearest winning neuron/node)

plt.figure(figsize=(10, 9))
plt.pcolormesh(newsom.distance_map(), cmap='Blues', alpha=.2)
plt.colorbar()

for num, c in enumerate(w_x[::1]):     # every 20th data point

    plt.scatter(w_y[num] + np.random.rand(1),   # add a random decimal to prevent overlapping of markers
                w_x[num] + np.random.rand(1),
                s=50, c='k', marker='x')

plt.title("IVT data across SOM lattice", fontsize=12)
plt.margins(x=0,y=0)
plt.grid()
plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
plt.show()

plt.figure(figsize=(10, 9))
frequencies = newsom.activation_response(data.values)
plt.pcolormesh(frequencies, cmap='Blues') 
plt.colorbar()
plt.title("data frequency (2d histogram) across SOM lattice", fontsize=12)
plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
plt.show()

# grabbing indices from SOM
#all the cases for 2016-2017 500 mb heights are fairly similar 
# create an empty dictionary using the rows and columns of SOM
keys = [i for i in product(range(som_grid_rows),range(som_grid_columns))]
winmap = {key: [] for key in keys}

# grab the indices for the data within the SOM lattice
for i, x in enumerate(data.values):
    winmap[newsom.winner(x)].append(i)

print(f"The rows and columns of the SOM lattice to use to grab SOM indexes:\n{[i for i in list(winmap.keys())]}")
som_keys = list(winmap.keys())
print(som_keys)
print(f"Number of composite maps: {len(som_keys)}")


##plot som##
fig, axs = plt.subplots(som_grid_rows, som_grid_columns, figsize=(16,12))
di_dates={}


##plot som##
#fig, geo_axs = plt.subplots(4, 4, figsize=(16,12))




# (1) loop through the SOM neurons, (2) grab data samples using SOM indices, (3) compute the mean for each SOM neuron, plot each 
for map_num in range(len(som_keys)):

#     
    temp_data = subsetarray[np.array(winmap[som_keys[map_num]])].unstack().mean(dim="time",skipna=True)
    # plot
    ht_data=height['variable'][np.array(winmap[som_keys[map_num]])].mean(dim="time",skipna=True).values
    label=axs[som_keys[map_num][0],som_keys[map_num][1]].pcolormesh(temp_data, cmap="viridis",vmin=200,vmax=850)
    htctr=axs[som_keys[map_num][0],som_keys[map_num][1]].contour(ht_data, colors="white",levels=[5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,6040,6100,6160,6220])
    #levels=[5280,5340,5400,5460,5520,558
    axs[som_keys[map_num][0],som_keys[map_num][1]].set_title(f"Sample size: {frequencies.flatten()[map_num]}", fontsize=12)
    axs[som_keys[map_num][0],som_keys[map_num][1]].clabel(htctr,htctr.levels[::2],fmt='%1.0f',colors='white',inline=True,inline_spacing=-3)


      


    n=(frequencies.flatten()[map_num])




plt.tight_layout()
fig = plt.gcf()
# twin_ax = fig.add_axes([-.05, 0.1, 0.02, 0.8])
# twin_ax.set_ylabel('Gridpoints in the north-south direction (1800 km)', fontsize=20, labelpad=10)
#twin_ax.yaxis.set_label_coords(1.1, 0.5)  # Adjust the position of the label
fig.text(-0.03, 0.5, 'Gridpoints in the south-north direction', fontsize=20, ha='center', va='center', rotation=90)


fig.text(.4,-.05,'Gridpoints in the west-east direction', fontsize=20, ha='center', va='center')

fig.suptitle("Current IVT Shaded \n 500-hPa heights contoured".format(sigma=newsom._sigma,lr =newsom._learning_rate), fontsize=20,y=1.05, x=0.45, fontweight='bold')

#Sigma:{sigma:.2f},LR:{lr:.4f}
#ax1=fig.colorbar(label, ax=fig.get_axes())
cbar=fig.colorbar(label, ax=fig.get_axes())

# cbar.ax.set_ylabel('Difference of IVT btw West-WRF - ERA5 (kg/ms)', rotation=270, fontsize=20,labelpad=20
#       )


cbar.ax.set_ylabel('IVT (kg/ms) shaded', rotation=270, fontsize=20,labelpad=23
      )
#plt.savefig('currentIVT_march6_2025.png',bbox_inches='tight',pad_inches = 0)
plt.show()


In [ ]:
#example SOM code after loading SOM:
lr_arr=[0.05,0.10]

sigma_arr=[2,2.5,3]

som_grid_rows=4
som_grid_columns=4

for l in lr_arr:
    
    for s in sigma_arr:
     
        #data2 = normalize_data(ht_anom_stack.values)
        data = normalize_data(subsetarray)
        # if som==newsom:
        # with open('som_current_mpas_dec5.p', 'wb') as outfile:
        #     pickle.dump(newsom, outfile)

        # som_grid_rows=4
        # som_grid_columns=4

        plt.figure(figsize=(10, 9))
        cs = plt.pcolormesh(newsom.distance_map(), cmap='hot')
        plt.title("distance map of the ivt", fontsize=12)
        plt.colorbar(cs)
        plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
        plt.show()


        # this may take a few minutes if lots of data! currently doing every 20th point so not bad

        # grab the x and y coords across the lattice for the winner node for each data point

        w_x, w_y = zip(*[newsom.winner(w) for w in data.values])
        print(len(w_x))
        w_x = np.array(w_x)
        w_y = np.array(w_y)
        print(w_x)
        print(w_y)

                # visualize where data falls in lattice (nearest winning neuron/node)

        plt.figure(figsize=(10, 9))
        plt.pcolormesh(newsom.distance_map(), cmap='Blues', alpha=.2)
        plt.colorbar()

        for num, c in enumerate(w_x[::1]):     # every 20th data point

            plt.scatter(w_y[num] + np.random.rand(1),   # add a random decimal to prevent overlapping of markers
                        w_x[num] + np.random.rand(1),
                        s=50, c='k', marker='x')

        plt.title("IVT data across SOM lattice", fontsize=12)
        plt.margins(x=0,y=0)
        plt.grid()
        plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
        plt.show()

        plt.figure(figsize=(10, 9))
        frequencies = newsom.activation_response(data.values)
        plt.pcolormesh(frequencies, cmap='Blues') 
        plt.colorbar()
        plt.title("data frequency (2d histogram) across SOM lattice", fontsize=12)
        plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
        plt.show()

        # grabbing indices from SOM
        #all the cases for 2016-2017 500 mb heights are fairly similar 
        # create an empty dictionary using the rows and columns of SOM
        keys = [i for i in product(range(som_grid_rows),range(som_grid_columns))]
        winmap = {key: [] for key in keys}

        # grab the indices for the data within the SOM lattice
        for i, x in enumerate(data.values):
            winmap[newsom.winner(x)].append(i)

        print(f"The rows and columns of the SOM lattice to use to grab SOM indexes:\n{[i for i in list(winmap.keys())]}")
        som_keys = list(winmap.keys())
        print(som_keys)
        print(f"Number of composite maps: {len(som_keys)}")


        ##plot som##
        fig, axs = plt.subplots(som_grid_rows, som_grid_columns, figsize=(16,12))
        di_dates={}

        # (1) loop through the SOM neurons, (2) grab data samples using SOM indices, (3) compute the mean for each SOM neuron, plot each 
        for map_num in range(len(som_keys)):

            #     
            temp_data = subsetarray[np.array(winmap[som_keys[map_num]])].unstack().mean(dim="time",skipna=True)
            ht_data=height['variable'][np.array(winmap[som_keys[map_num]])].mean(dim="time",skipna=True).values
            label=axs[som_keys[map_num][0],som_keys[map_num][1]].pcolormesh(temp_data, cmap="viridis",vmin=200,vmax=850)
            htctr=axs[som_keys[map_num][0],som_keys[map_num][1]].contour(ht_data, colors="white",levels=[5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,6040,6100,6160,6220])
            #levels=[5280,5340,5400,5460,5520,558
            axs[som_keys[map_num][0],som_keys[map_num][1]].set_title(f"Sample size: {frequencies.flatten()[map_num]}", fontsize=12)
            axs[som_keys[map_num][0],som_keys[map_num][1]].clabel(htctr,htctr.levels[::2],fmt='%1.0f',colors='white',inline=True,inline_spacing=-3)





            n=(frequencies.flatten()[map_num])




        plt.tight_layout()
        fig = plt.gcf()
        # twin_ax = fig.add_axes([-.05, 0.1, 0.02, 0.8])
        # twin_ax.set_ylabel('Gridpoints in the north-south direction (1800 km)', fontsize=20, labelpad=10)
        #twin_ax.yaxis.set_label_coords(1.1, 0.5)  # Adjust the position of the label
        fig.text(-0.03, 0.5, 'Gridpoints in the south-north direction', fontsize=20, ha='center', va='center', rotation=90)


        fig.text(.4,-.05,'Gridpoints in the west-east direction', fontsize=20, ha='center', va='center')

        fig.suptitle("Current IVT Shaded \n 500-hPa heights contoured Sigma:{sigma:.2f},LR:{lr:.2f}".format(sigma=newsom._sigma,lr =newsom._learning_rate), fontsize=20,y=1.05, x=0.45, fontweight='bold')

        #Sigma:{sigma:.2f},LR:{lr:.4f}
        #ax1=fig.colorbar(label, ax=fig.get_axes())
        cbar=fig.colorbar(label, ax=fig.get_axes())

        # cbar.ax.set_ylabel('Difference of IVT btw West-WRF - ERA5 (kg/ms)', rotation=270, fontsize=20,labelpad=20
        #       )


        cbar.ax.set_ylabel('IVT (kg/ms) shaded', rotation=270, fontsize=20,labelpad=23
          )
        #plt.savefig('currentIVT_march6_2025.png',bbox_inches='tight',pad_inches = 0)
        plt.show()


In [ ]:
# weights.shape

In [ ]:
# feature_names=['weight_1','weight_2','weight_3','weight_4']
# weights = som.get_weights()  # shape: (dimension1, dimension2, num_features)

# plt.figure(figsize=(10, 10))

# for i in range(weights.shape[1]):  # loop over each feature
#     plt.subplot(2, 2, i+1)
#     plane = weights[:, :, i].T  # transpose for correct orientation
#     plt.imshow(plane, cmap='coolwarm', origin='lower')
#     plt.title(f'Weights for {feature_names[i]}')
#     plt.colorbar()
#     plt.xticks([]), plt.yticks([])

# plt.tight_layout()
# plt.show()

In [ ]:
#example SOM code after loading SOM:
lr_arr=[0.05,0.10]

sigma_arr=[2,2.5,3]

som_grid_rows=4
som_grid_columns=4

for l in lr_arr:
    
    for s in sigma_arr:
        
        som_grid_rows = 4           # (y-axis)
        som_grid_columns = 4        # (x-axis)
        input_length = subsetarray.shape[1]    # using preprocessed data array; Number of the elements of the vectors in input.
        sigma = s               # Spread of the neighborhood function, needs to be adequate to the dimensions of the map.
        learning_rate = l       # initial learning rate (at the iteration t we have learning_rate(t) = learning_rate / (1 + t/T) where T is #num_iteration/2)
        decay_function = asymptotic_decay

        """Function that reduces learning_rate and sigma at each iteration
        the default function is (asymptotic_decay):
                    learning_rate / (1+t/(max_iterarations/2))

        A custom decay function will need to to take in input
        three parameters in the following order:

        1. learning rate
        2. current iteration
        3. maximum number of iterations allowed

        Note that if a lambda function is used to define the decay
        MiniSom will not be pickable anymore."""

        neighborhood_function = 'gaussian'

        """Function that weights the neighborhood of a position in the map. 
        Possible values: 'gaussian', 'mexican_hat', 'bubble', 'triangle', 
        which takes in sigma."""

        topology = 'rectangular'                 # Topology of the map; Possible values: 'rectangular', 'hexagonal'
        activation_distance = 'euclidean'        # Distance used to activate the map; Possible values: 'euclidean', 'cosine', 'manhattan', 'chebyshev'
        random_seed = 1                          # Random seed to use for reproducibility. Using 1.
        
        som = MiniSom(
            som_grid_rows,
            som_grid_columns,
            input_length,
            sigma,
            learning_rate,
            decay_function,
            neighborhood_function,
            topology,
            activation_distance,
            random_seed) 
        
        data = normalize_data(subsetarray)
        num_iteration = 100000
        random_order = True
        verbose = True
        data.shape
        


        som.pca_weights_init(data.values) 
        
     
        #data2 = normalize_data(ht_anom_stack.values)
        data = normalize_data(subsetarray)
        # if som==newsom:
        # with open('som_current_mpas_dec5.p', 'wb') as outfile:
        #     pickle.dump(newsom, outfile)

        # som_grid_rows=4
        # som_grid_columns=4
        
        som.train(
            data.values,
            num_iteration,
            random_order,
            verbose)

        plt.figure(figsize=(10, 9))
        cs = plt.pcolormesh(newsom.distance_map(), cmap='hot')
        plt.title("distance map of the ivt", fontsize=12)
        plt.colorbar(cs)
        plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
        plt.show()


        # this may take a few minutes if lots of data! currently doing every 20th point so not bad

        # grab the x and y coords across the lattice for the winner node for each data point

        w_x, w_y = zip(*[newsom.winner(w) for w in data.values])
        print(len(w_x))
        w_x = np.array(w_x)
        w_y = np.array(w_y)
        print(w_x)
        print(w_y)

                # visualize where data falls in lattice (nearest winning neuron/node)

        plt.figure(figsize=(10, 9))
        plt.pcolormesh(newsom.distance_map(), cmap='Blues', alpha=.2)
        plt.colorbar()

        for num, c in enumerate(w_x[::1]):     # every 20th data point

            plt.scatter(w_y[num] + np.random.rand(1),   # add a random decimal to prevent overlapping of markers
                        w_x[num] + np.random.rand(1),
                        s=50, c='k', marker='x')

        plt.title("IVT data across SOM lattice", fontsize=12)
        plt.margins(x=0,y=0)
        plt.grid()
        plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
        plt.show()

        plt.figure(figsize=(10, 9))
        frequencies = newsom.activation_response(data.values)
        plt.pcolormesh(frequencies, cmap='Blues') 
        plt.colorbar()
        plt.title("data frequency (2d histogram) across SOM lattice", fontsize=12)
        plt.ylim(som_grid_rows, 0)  # flip the y axis to be the same as composite map axes later
        plt.show()

        # grabbing indices from SOM
        #all the cases for 2016-2017 500 mb heights are fairly similar 
        # create an empty dictionary using the rows and columns of SOM
        keys = [i for i in product(range(som_grid_rows),range(som_grid_columns))]
        winmap = {key: [] for key in keys}

        # grab the indices for the data within the SOM lattice
        for i, x in enumerate(data.values):
            winmap[newsom.winner(x)].append(i)

        print(f"The rows and columns of the SOM lattice to use to grab SOM indexes:\n{[i for i in list(winmap.keys())]}")
        som_keys = list(winmap.keys())
        print(som_keys)
        print(f"Number of composite maps: {len(som_keys)}")


        ##plot som##
        fig, axs = plt.subplots(som_grid_rows, som_grid_columns, figsize=(16,12))
        di_dates={}

        # (1) loop through the SOM neurons, (2) grab data samples using SOM indices, (3) compute the mean for each SOM neuron, plot each 
        for map_num in range(len(som_keys)):

            #     
            temp_data = subsetarray[np.array(winmap[som_keys[map_num]])].unstack().mean(dim="time",skipna=True)
            ht_data=height['variable'][np.array(winmap[som_keys[map_num]])].mean(dim="time",skipna=True).values
            label=axs[som_keys[map_num][0],som_keys[map_num][1]].pcolormesh(temp_data, cmap="viridis",vmin=200,vmax=850)
            htctr=axs[som_keys[map_num][0],som_keys[map_num][1]].contour(ht_data, colors="white",levels=[5280,5340,5400,5460,5520,5580,5640,5700,5760,5820,5880,6040,6100,6160,6220])
            #levels=[5280,5340,5400,5460,5520,558
            axs[som_keys[map_num][0],som_keys[map_num][1]].set_title(f"Sample size: {frequencies.flatten()[map_num]}", fontsize=12)
            axs[som_keys[map_num][0],som_keys[map_num][1]].clabel(htctr,htctr.levels[::2],fmt='%1.0f',colors='white',inline=True,inline_spacing=-3)





            n=(frequencies.flatten()[map_num])




        plt.tight_layout()
        fig = plt.gcf()
        # twin_ax = fig.add_axes([-.05, 0.1, 0.02, 0.8])
        # twin_ax.set_ylabel('Gridpoints in the north-south direction (1800 km)', fontsize=20, labelpad=10)
        #twin_ax.yaxis.set_label_coords(1.1, 0.5)  # Adjust the position of the label
        fig.text(-0.03, 0.5, 'Gridpoints in the south-north direction', fontsize=20, ha='center', va='center', rotation=90)


        fig.text(.4,-.05,'Gridpoints in the west-east direction', fontsize=20, ha='center', va='center')

        fig.suptitle("Current IVT Shaded \n 500-hPa heights contoured Sigma:{sigma:.2f},LR:{lr:.2f}".format(sigma=newsom._sigma,lr =newsom._learning_rate), fontsize=20,y=1.05, x=0.45, fontweight='bold')

        #Sigma:{sigma:.2f},LR:{lr:.4f}
        #ax1=fig.colorbar(label, ax=fig.get_axes())
        cbar=fig.colorbar(label, ax=fig.get_axes())

        # cbar.ax.set_ylabel('Difference of IVT btw West-WRF - ERA5 (kg/ms)', rotation=270, fontsize=20,labelpad=20
        #       )


        cbar.ax.set_ylabel('IVT (kg/ms) shaded', rotation=270, fontsize=20,labelpad=23
          )
        #plt.savefig('currentIVT_march6_2025.png',bbox_inches='tight',pad_inches = 0)
        plt.show()
